In [ ]:
from system.plan_enumeration.subproblems import Subproblem
from typing import Type
from system.plan_enumeration.plan_table import (
    PlanTable,
    SizeBasedPlanTable,
    StandardPlanTable,
)
from system.plan_enumeration.join_graph import (
    JoinGraph,
    JoinGraphFactory,
    StarQueryFactory,
    ChainQueryFactory,
    CliqueQueryFactory,
    CycleQueryFactory,
)
from system.plan_enumeration.cost_function import CostFunction, C_Out
from system.plan_enumeration.utils import (
    randomize_cardinalities_and_enumerate_join_graph,
    randomize_cardinalities,
    print_results,
)
from system.plan_enumeration.cardinality_table import CardinalityTable
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod

# Plan Enumeration
In this notebook, we want to analyze three different algorithms (DPsize, DPsub, and DPccp) that try to enumerate the search space of bushy plans without Cartesian Products. That means, we want to study algorithms implementing the following interface:

In [ ]:
class PlanEnumerator(ABC):
    """
    Abstract enumerator class to enumerate the best join plan for a given join graph.
    """

    @abstractmethod
    def enumerate(
        self,
        join_graph: JoinGraph,
        cardinality_table: CardinalityTable,
        cost_function: CostFunction,
        print_info: bool = False,
    ) -> tuple[int, tuple, int]:
        """
        Computes and returns the best join order for the given problem sizes.
        :param join_graph: The join graph.
        :param cardinality_table: The cardinality table to be used.
        :param cost_function: A cost function determining the cost of different operations.
        :param print_info: A flag indicating whether additional info should be printed during enumeration.
        :return: A tuple, consisting of the costs of the best plan, the best plan as a tuple and the number of pairs that were enumerated.
        """
        pass

All algorithms will have in common that they utilize a so-called plan table, which will the best plans found so far for a given subproblems. This allows us to make use of dynamic programming, which greatly enhances the performance of the presented algorithms. 

## DPsize
DPsize is a well-known dynamic programing algorithm to enumerate the entire search space of bushy plans without Cartesian Products. For that, it uses a size-based
subproblem enumeration, i.e. it enumerates subproblems of increasing sizes. During the enumeration, it does not consider the
actual join graph, which means that Cartesian Products, as well as disconnected or overlapping join pairs might be enumerated, which need to be filtered out.

In [ ]:
class DP_Size(PlanEnumerator):
    """
    DPsize uses a size-based subproblem enumeration, i.e. it enumerates subproblems of increasing sizes. During the enumeration, it does
    not consider the actual join graph, which means that Cartesian Products, as well as disconnected join pairs might be enumerated, which need
    to be filtered out.
    """

    def enumerate(
        self,
        join_graph: JoinGraph,
        cardinality_table: CardinalityTable,
        cost_function: CostFunction,
        print_info: bool = False,
    ) -> tuple[int, tuple, int]:
        """
        Computes and returns the best join order for the given problem sizes.
        :param join_graph: The join graph.
        :param cardinality_table: The cardinality table to be used.
        :param cost_function: A cost function determining the cost of different operations.
        :param print_info: A flag indicating whether additional info should be printed during enumeration.
        :return: A tuple, consisting of the costs of the best plan, the best plan as a tuple and the number of pairs that were enumerated.
        """
        # Initialize plan table with the join graph
        # A size plan table stores entries according to their subproblem size, which
        # allows iterating through existing subproblems based on their size
        plan_table: SizeBasedPlanTable = SizeBasedPlanTable(
            join_graph, cost_function, cardinality_table
        )

        # Count number of pairs that were enumerated
        enumerated_pairs: int = 0
        # Count number of valid pairs
        valid_pairs: int = 0

        # Go through each pair generated by the enumerator below
        # It still needs to make sure that both subproblems are actually
        # connected and that they are disjoint.
        for s1, s2 in self.__generate_pairs_size_based(join_graph, plan_table):
            assert s1 in plan_table
            assert s2 in plan_table
            enumerated_pairs += 1
            # Check whether s1 and s2 are disjoint. Pairs which are not disjoint
            # are no valid pairs. They also need to be connected in the join graph.
            if not s1.overlaps(s2) and join_graph.are_connected(s1, s2):
                valid_pairs += 1
                # Update the table with the newly generated pair
                plan_table.update_table(s1, s2, cardinality_table, cost_function)
                if print_info:
                    print(f"Enumerated valid pair {s1.as_set()}, {s2.as_set()}.")
            else:
                if print_info:
                    print(f"Enumerated invalid pair {s1.as_set()}, {s2.as_set()}.")

        complete_problem: Subproblem = Subproblem.get_complete_problem(len(join_graph))

        if print_info:
            print(
                f"{enumerated_pairs} pairs were enumerated, out of which {valid_pairs} were valid!"
            )

        return (
            plan_table.get_costs_for_subproblem(complete_problem),
            plan_table.get_plan_for_subproblem(complete_problem),
            enumerated_pairs,
        )

    def __generate_pairs_size_based(
        self, join_graph: JoinGraph, plan_table: SizeBasedPlanTable
    ):
        """
        Enumerate through the search space by utilizing the size of subproblems.
        :param join_graph: The join graph.
        :param plan_table: The underlying plan table.
        """
        num_nodes: int = len(join_graph.adjacency_matrix)
        # Enumerate all possible subproblem sizes starting from 2
        for main_size in range(2, num_nodes + 1):
            # Enumerate all sizes that are smaller than size_1 // 2
            # to avoid duplicate work.
            for s1_size in range(1, (main_size // 2) + 1):
                s2_size: int = main_size - s1_size

                # If both sizes are equal, you need to evaluate symmetric pairs only once
                if s1_size == s2_size:
                    # Enumerate all subproblems of size s1_size
                    for i, s1 in enumerate(plan_table.entries_with_size(s1_size)):
                        # Evaluated entries can be addressed by an index
                        # Enumerate all subproblems of size s2_size, and start with index i + 1
                        # to avoid having to re-enumerate symmetric pairs
                        for s2 in plan_table.entries_with_size(s2_size, i + 1):
                            # New pair was generated
                            yield s1, s2
                else:
                    # Enumerate all subproblems of size s1_size
                    for s1 in plan_table.entries_with_size(s1_size):
                        # Enumerate all subproblems of size s2_size
                        for s2 in plan_table.entries_with_size(s2_size):
                            # New pair was generated
                            yield s1, s2

In the following you can see an example of which pairs DPsub would enumerate for a given query (in our cases a star query with 5 nodes), and which of them are valid pairs, and which are not, i.e. which pairs can not be used to update the plan table. Note that the underlying cardinality table is completely randomized in this example. 


In [ ]:
# Construct a cycle query with 4 nodes. Therefore, this query has the following structure:
# 0 -- 1
# |    |
# 2 -- 3
cycle_query = CycleQueryFactory.construct_join_graph(4)
# Define a Cardinality Table, i.e. a table estimating the sizes of subproblems.
# For this demonstration, we just randomize them, i.e. they do not represent realistic data by any means
cardinality_table: CardinalityTable = CardinalityTable()
randomize_cardinalities(cardinality_table, 4, 42)
# Start the enumeration, use C_out as the underlying cost function
results = DP_Size().enumerate(
    join_graph=cycle_query,
    cardinality_table=cardinality_table,
    cost_function=C_Out(),
    print_info=True,
)
print_results(results)

As you can see, the majority of plans that was enumerated was actually invalid, i.e. a lot of work was done unnecessarily. Maybe an algorithm that enumerated directly over all possible subproblems could yield a better performance. 

## DPsub  
Another widely known enumeration algorithm is DPsub. Instead of enumerating problems based on their size, it generates all possible pairs
for each possible subproblem. It suffers from similar issues as DPsize, as it still generates Cartesian Products. However, no overlapping pairs will be generated at any point.

In [ ]:
class DP_Sub(PlanEnumerator):
    """
    DPsub generates all possible pairs for each connected subproblem. It still generates Cartesian Products, that need to be filtered out.
    """

    def enumerate(
        self,
        join_graph: JoinGraph,
        cardinality_table: CardinalityTable,
        cost_function: CostFunction,
        print_info: bool = False,
    ) -> tuple[int, tuple, int]:
        """
        Computes and returns the best join order for the given problem sizes.
        :param join_graph: The join graph.
        :param cardinality_table: The cardinality table to be used.
        :param cost_function: A cost function determining the cost of different operations.
        :param print_info: A flag indicating whether additional info should be printed during enumeration.
        :return: A tuple, consisting of the costs of the best plan, the best plan as a tuple and the number of pairs that were enumerated.
        """
        # Initialize plan table with the join graph
        plan_table: PlanTable = StandardPlanTable(
            join_graph, cost_function, cardinality_table
        )

        # Count number of pairs that were enumerated
        enumerated_pairs: int = 0
        # Count number of valid pairs
        valid_pairs: int = 0

        # Go through each generated pair and make sure that both problems s1 and s2
        # are connected
        for s1, s2 in self.__generate_pairs_sub_based(join_graph):
            enumerated_pairs += 1
            # s1 | s2 is guaranteed to be connected as this point
            assert join_graph.is_connected(s1 | s2)
            # s1 and s2 are guaranteed to be disjoint here
            assert not s1.overlaps(s2)
            # We need to check whether s1 and s2 themselves are connected
            # s1 and s2 are connected if they are contained in the plan table
            # meaning they were enumerated as a valid subproblem at an earlier point
            if s1 in plan_table and s2 in plan_table:
                valid_pairs += 1
                # Update the table with the newly generated pair
                plan_table.update_table(s1, s2, cardinality_table, cost_function)
                if print_info:
                    print(f"Enumerated valid pair {s1.as_set()}, {s2.as_set()}")
            else:
                if print_info:
                    print(f"Enumerated invalid pair {s1.as_set()}, {s2.as_set()}")

        complete_problem: Subproblem = Subproblem.get_complete_problem(len(join_graph))

        if print_info:
            print(
                f"{enumerated_pairs} pairs were enumerated, out of which {valid_pairs} were valid"
            )

        return (
            plan_table.get_costs_for_subproblem(complete_problem),
            plan_table.get_plan_for_subproblem(complete_problem),
            enumerated_pairs,
        )

    def __generate_pairs_sub_based(self, join_graph: JoinGraph):
        """
        Generate the required pairs by enumerating all possible subproblems.
        :param join_graph: The join graph.
        """
        num_nodes: int = len(join_graph.adjacency_matrix)
        # Enumerate all subproblems by increasing the underlying bitset
        for i in range(1, 1 << num_nodes):
            # Find the best plan for this problem
            main_problem: Subproblem = Subproblem(i)

            # Singletons do not have to be considered
            if len(main_problem) == 1:
                continue

            # The problem also needs to be connected, otherwise
            # it would not be a valid subproblem
            if not join_graph.is_connected(main_problem):
                continue

            # Again, there is no need to enumerate symmetric pair twice, which can be achieved
            # by stopping the enumeration after the first half is done.
            # Setting the stop problem exactly to the singleton pointing to the
            # position of the MSB stops the enumeration exactly at this point
            stop_problem: Subproblem = main_problem.get_msb_problem()
            # Start the enumeration with the smallest singleton of the main problem, stop when the stop problem has been reached
            for s1 in Subproblem.enumerate_all_subproblems(
                main_problem, stop_problem=stop_problem
            ):
                # Generate the corresponding partner subset
                # Given that s1 | s2 == main_problem, we know s1 and s2 are connected via at least one join edge
                s2: Subproblem = main_problem - s1
                # New pair was generated
                yield s1, s2

Again, we can check the enumeration with this new algorithm.

In [ ]:
results = DP_Sub().enumerate(
    join_graph=cycle_query,
    cardinality_table=cardinality_table,
    cost_function=C_Out(),
    print_info=True,
)
print_results(results)

Even though DPsub manages to enumerate less invalid pairs than DPsize for this cycle query, it still enumerates unnecessary plans. This is because both algorithms do not actually make use of the join graph to enumerate their plans, they just use to it verify already enumerated plans. Therefore, a scheme that actually uses the join graph directly should perform much better.

## Using the join graph for the enumeration
A first important step would be to only enumerate connected subproblems. The overall idea is that we want to start with the singleton relations and recursively extend them with their neighbors. However, a naive implementation would create many duplicates, which is of course unnecessary work that we should try to avoid, requiring a more strategic traversal of the graph. For that, we first give each node a certain order by numerating each node in the graph in a BFS manner (automatically done in our join graph factories). The start node does not matter here. The general idea for our optimization will be that we only want to recursively extend a set S with neighbors which have a higher order than the smallest relation in the set S. For example, applied on the example above, we would only extend the start problem {2} with its neighbor {3}. However, the start problem {3} would not be extended with any neighbor, as all neighbors have a smaller order. This way we can ensure that subproblems are gradually extended with the neighbors without creating any duplicates.

 To achieve that algorithmically, we introduce a dedicated exclude set X. The idea behind that is that we only want to recursively extend the current set S with neighbors which do not belong to X. X will therefore always contain relations, whose BFS numbering is smaller (or equal) than the smallest relation in S.
  
 Additionally, when evaluating a recursive call to a child set S' of S, we also want to add the neighbors of S to X. This causes the child set {0, 1, 3} to not be extended with {2}, as {2} is already a neighbor of {0}, which is a parent of {0, 1, 3}. Instead, {0, 1, 2, 3} would be constructed by extending {0, 1, 2} with {3}. 
 
This entire procedure requires little changes to the code above.

In [ ]:
def generate_connected_subproblems(join_graph: JoinGraph):
    """
    Generate all connected subproblems of the join graph.
    :param join_graph: The join graph.
    """
    complete_problem: Subproblem = Subproblem.get_complete_problem(len(join_graph))

    # Top-level exclude set, always ignore neighbors whose BFS number is <= than the current singleton.
    exclude: Subproblem = complete_problem
    # Traverse all singleton sets as starting point. Go from the highest to the lowest.
    for singleton_problem in reversed(complete_problem):
        yield singleton_problem
        yield from generate_recursively(join_graph, singleton_problem, exclude)
        # Current problem can now be considered
        exclude -= singleton_problem


def generate_recursively(
    join_graph: JoinGraph,
    start: Subproblem,
    exclude: Subproblem,
):
    """
    Recursively extends the given start_problems with its neighbors. Ignore relations contained in the exclude_set.
    :param join_graph: The join graph.
    :param start: The start problem.
    :param exclude: The exclude problem.
    """
    # Get direct neighbors of the subproblem, ignore those from the exclude set
    neighbors: Subproblem = join_graph.get_neighbors(start) - exclude

    # Get all subproblems contained in the neighbor set
    neighbor_subproblems: list[Subproblem] = Subproblem.get_all_subproblems(neighbors)

    # Combine all neighbor subproblems with the started subproblem
    for neighbor_subproblem in neighbor_subproblems:
        yield start | neighbor_subproblem
    for neighbor_subproblem in neighbor_subproblems:
        # Proceed recursively, extend the exclude set with the current neighbors
        yield from generate_recursively(
            join_graph,
            start | neighbor_subproblem,
            exclude | neighbors,
        )

Note that we no longer need a termination condition in our recursive call, as this is now implicitly handled by our exclusion set. Calling the function now yields the following result:

In [ ]:
for subproblem in generate_connected_subproblems(cycle_query):
    print(f"{subproblem}")

As you can see, we have not enumerated a single duplicate. Great! But one problem remains: How can we actually generate the required join pairs? Using the strategies that we have seen in DPsize/DPsub would result in similar problems again. What we need 
to achieve is the following: 
1. All valid (i.e. disjoint, connected) pairs need to be enumerated.
2. No invalid plan must be enumerated at any point. 
3. When generating a new pair (S1, S2), the best plans for S1 and S2 must already be evaluated. This is a core requirement to make use of dynamic programming.

For that, we want to use follow the following strategy:

We enumerate all connected subgraphs (csg's) as described above. Whenever we enumerate a new csg S1, we enumerate complementing subgraphs (cmp's) S2 using the same strategy, such that the csg-cmp (ccp) pair (S1, S2) is a valid pair, i.e. S1 and S2 are connected and have been evaluated already. Obviously, we can not just enumerate all cmp's directly for each csg, as we would violate Conditions 2 and 3 above with the strategy. To solve this problem, we can make use of the following observation:

During our enumeration we only extend our subproblem sets with nodes whose BFS number is higher than the smallest in the current subproblem. This is also reflected in the order in which pairs are enumerated! Therefore, we need to make sure, that we ignore cmp problems containing nodes whose number is smaller than the smallest in the current subproblem. Therefore, we get the following function, that generates all ccp's in a correct order:


In [ ]:
def generate_ccp_pairs(join_graph: JoinGraph):
    """
    Enumerate all ccp pairs of the given join graph.
    :param join_graph: The join graph.
    """
    for csg_problem in generate_connected_subproblems(join_graph):
        # We do not want the subproblem to be contained in any cmp problem at any problem
        # We also always want to ignore cmp problems that contain relations whose number is smaller than the smallest in the csg
        always_exclude_set: Subproblem = csg_problem | csg_problem.get_b_min()
        # Start the enumeration of the cmp problems with the direct neighbors of the csg problem, ignore csg problem
        csg_neighbors: Subproblem = (
            join_graph.get_neighbors(csg_problem) - always_exclude_set
        )
        # Define an additional top-level ignore set, we still want to ignore all subproblems with smaller/equal numbers than the current subproblem
        # This time, however, we only enumerate the direct neighbors of the csg, thus we need to gradually reduce the neighbor set, instead of the whole subproblem.
        exclude_set: Subproblem = csg_neighbors

        # Again traverse all singleton problems in reversed order
        for singleton_problem in reversed(csg_neighbors):
            yield csg_problem, singleton_problem
            # The exclude_set is gradually reduced, we still need to make sure that we never include a relation from the always_exclude_set
            # Make use of the already defined recursive function, as the required behavior here is exactly the same.
            for cmp_problem in generate_recursively(
                join_graph,
                singleton_problem,
                exclude_set | always_exclude_set,
            ):
                yield csg_problem, cmp_problem
            # Current problem can now be considered
            exclude_set -= singleton_problem

Let's give this function a try!

In [ ]:
for ccp_pair in generate_ccp_pairs(cycle_query):
    print(f"Enumerated ccp pair ({ccp_pair[0].as_set()}, {ccp_pair[1].as_set()})")

This functions enables the enumeration in the way we want it. We never enumerate a symmetric pair more than once, and every time a pair is generated we also make sure that all its subproblems were enumerated already! Awesome!! This procedure is then the entire idea behind DPccp, so let's give the final enumeration algorithm:

## DPccp
DPsize and DPsub suffer from the problem that invalid problem pairs might be generated that need to be filtered out. While
they are not used to actually update the plan table, the overhead caused by these unnecessary enumerations will be highly noticeable. The
algorithm presented below, DP_CPP, fixes this issue by only enumerating connected pairs, where both underlying subproblems are also connected.
Unlike the schemes presented before, it utilizes the join graph for this purpose.

In [ ]:
class DP_CCP(PlanEnumerator):
    """
    Uses the join graph to enumerate all ccp pairs, without enumerating any invalid pair.
    """

    def enumerate(
        self,
        join_graph: JoinGraph,
        cardinality_table: CardinalityTable,
        cost_function: CostFunction,
        print_info: bool = False,
    ) -> tuple[int, tuple, int]:
        """
        Computes and returns the best join order for the given problem sizes.
        :param join_graph: The join graph.
        :param cardinality_table: The cardinality table to be used.
        :param cost_function: A cost function determining the cost of different operations.
        :param print_info: A flag indicating whether additional info should be printed during enumeration.
        :return: A tuple, consisting of the costs of the best plan, the best plan as a tuple and the number of pairs that were enumerated.
        """
        # Initialize plan table with the join graph
        plan_table: PlanTable = StandardPlanTable(
            join_graph, cost_function, cardinality_table
        )

        # Count number of pairs that were enumerated
        enumerated_pairs: int = 0

        # Use previously defined generator to iterator through each ccp
        # Note that no checks are required here anymore, as the ccp is guaranteed to be valid
        for s1, s2 in generate_ccp_pairs(join_graph):
            enumerated_pairs += 1

            if print_info:
                print(f"Enumerated valid pair {s1.as_set()}, {s2.as_set()}")

            # Update the table with the newly generated pair
            plan_table.update_table(s1, s2, cardinality_table, cost_function)

        complete_problem: Subproblem = Subproblem.get_complete_problem(len(join_graph))

        if print_info:
            print(f"Enumerated {enumerated_pairs} valid pairs")

        return (
            plan_table.get_costs_for_subproblem(complete_problem),
            plan_table.get_plan_for_subproblem(complete_problem),
            enumerated_pairs,
        )

Lets run our example graph one last time with the new enumerator: 

In [ ]:
results = DP_CCP().enumerate(
    join_graph=cycle_query,
    cardinality_table=cardinality_table,
    cost_function=C_Out(),
    print_info=True,
)
print_results(results)

As you can see, we still generated the same plan, and still generated all 18 valid pairs, but without generating any invalid or duplicate plan. That is awesome! But how much gain can we achieve by that?

## Evaluate the algorithms on different graph shapes
Having said all this, the decreased number of unnecessary pairs that are generated should create a great benefit in our enumerations. Below you can find a small benchmark showcasing the difference in enumeration time for different graph shapes. As predicted DP_CCP is the best algorithm overall, having the best or nearly the best performance, for all graph shapes. The only exception to that rule is clique graphs, where DP_Sub is clearly the best. However, this is not a big issue, given that clique graphs have no practical relevance.


In [ ]:
import time

num_nodes: list[int] = list(range(5, 13))


def evaluate_algorithms_on_graphs(graph_factory: Type[JoinGraphFactory]):
    """
    Evaluate all enumeration algorithms on different graph sizes.
    :param graph_factory: A static class type used to define methods to create certain graphs.
    """
    # Store number of enumerated pairs and the required time
    result_dict: dict[str, list[int]] = {
        algorithm: [] for algorithm in ["DP_CCP", "DP_Sub", "DP_Size"]
    }
    method_dict: dict[str, PlanEnumerator] = {
        "DP_CCP": DP_CCP(),
        "DP_Sub": DP_Sub(),
        "DP_Size": DP_Size(),
    }

    # Do an experiment for each node size
    for num in num_nodes:
        # Perform the experiment for each algorithm and store the result in the dictionary from above
        for algorithm, algorithm_result_lists in result_dict.items():
            start_time = time.time()
            _, _, enumerated_pairs = randomize_cardinalities_and_enumerate_join_graph(
                graph_factory.construct_join_graph(num), method_dict[algorithm]
            )
            end_time = time.time()
            algorithm_result_lists.append(end_time - start_time)

    # Plot data
    x = np.arange(len(num_nodes))  # the label locations
    fig, ax = plt.subplots(layout="constrained", figsize=(4, 3))
    for shape, shape_list in result_dict.items():
        ax.plot(x, shape_list, label=shape)

    ax.set_ylabel("Required Enumeration time in s")
    ax.set_xlabel("# of Nodes")
    ax.set_xticks(x, num_nodes)
    fig.legend(bbox_to_anchor=(0.2, 0.99), loc="upper left")

### Chain Query

In [ ]:
evaluate_algorithms_on_graphs(ChainQueryFactory)

### Star Query

In [ ]:
evaluate_algorithms_on_graphs(StarQueryFactory)

### Cycle Query

In [ ]:
evaluate_algorithms_on_graphs(CycleQueryFactory)

### Clique Query

In [ ]:
evaluate_algorithms_on_graphs(CliqueQueryFactory)

# Unittests
Always test your code!


In [ ]:
import unittest


class EnumeratorTests(unittest.TestCase):
    def test_plan_enumerators(self):
        join_graph: JoinGraph = ChainQueryFactory.construct_join_graph(5)
        cardinality_table: CardinalityTable = CardinalityTable()
        # Set cardinalities for important subproblems
        # Cardinalities for Size 1
        cardinality_table.update_cardinality_estimation(Subproblem(0b1), 5000)
        cardinality_table.update_cardinality_estimation(Subproblem(0b10), 4000)
        cardinality_table.update_cardinality_estimation(Subproblem(0b100), 500)
        cardinality_table.update_cardinality_estimation(Subproblem(0b1000), 100)
        cardinality_table.update_cardinality_estimation(Subproblem(0b10000), 1000)
        # Cardinalities for Size 2
        cardinality_table.update_cardinality_estimation(Subproblem(0b11), 100)
        cardinality_table.update_cardinality_estimation(Subproblem(0b110), 500)
        cardinality_table.update_cardinality_estimation(Subproblem(0b1100), 100)
        cardinality_table.update_cardinality_estimation(Subproblem(0b11000), 100)
        # Cardinalities for Size 3
        cardinality_table.update_cardinality_estimation(Subproblem(0b111), 500)
        cardinality_table.update_cardinality_estimation(Subproblem(0b1110), 50)
        cardinality_table.update_cardinality_estimation(Subproblem(0b11100), 100)
        # Cardinalities for Size 4
        cardinality_table.update_cardinality_estimation(Subproblem(0b1111), 10)
        cardinality_table.update_cardinality_estimation(Subproblem(0b11110), 100)
        # Cardinalities for Size 5
        cardinality_table.update_cardinality_estimation(Subproblem(0b11111), 10)

        best_plan: tuple = (
            (((3, 2), 1), 0),
            4,
        )
        best_costs: int = 10770
        algorithm_pairs: dict[str, int] = {
            "DP_Sub": 42,
            "DP_Size": 73,
            "DP_CCP": 20,
        }

        def _test_plan_costs(enumeration_algorithm: PlanEnumerator):
            costs, plan, enumerated_pairs = enumeration_algorithm.enumerate(
                join_graph, cardinality_table, C_Out(), False
            )
            self.assertEqual(plan, best_plan)
            self.assertEqual(costs, best_costs)
            self.assertEqual(
                enumerated_pairs,
                algorithm_pairs[enumeration_algorithm.__class__.__name__],
            )

        _test_plan_costs(DP_Size())
        _test_plan_costs(DP_Sub())
        _test_plan_costs(DP_CCP())

    def test_csg_enumeration(self):
        chain_query: JoinGraph = ChainQueryFactory.construct_join_graph(5)
        correct_csgs: list[Subproblem] = [
            Subproblem(0b10000),
            Subproblem(0b1000),
            Subproblem(0b11000),
            Subproblem(0b100),
            Subproblem(0b1100),
            Subproblem(0b11100),
            Subproblem(0b10),
            Subproblem(0b110),
            Subproblem(0b1110),
            Subproblem(0b11110),
            Subproblem(0b1),
            Subproblem(0b11),
            Subproblem(0b111),
            Subproblem(0b1111),
            Subproblem(0b11111),
        ]

        csgs_without_duplicates: list[Subproblem] = list(
            generate_connected_subproblems(chain_query)
        )

        self.assertListEqual(csgs_without_duplicates, correct_csgs)

    def test_ccp_helper_enumeration(self):
        chain_query: JoinGraph = ChainQueryFactory.construct_join_graph(5)

        correct_ccps: list[tuple[Subproblem, Subproblem]] = [
            (Subproblem(0b1000), Subproblem(0b10000)),
            (Subproblem(0b100), Subproblem(0b1000)),
            (Subproblem(0b100), Subproblem(0b11000)),
            (Subproblem(0b1100), Subproblem(0b10000)),
            (Subproblem(0b10), Subproblem(0b100)),
            (Subproblem(0b10), Subproblem(0b1100)),
            (Subproblem(0b10), Subproblem(0b11100)),
            (Subproblem(0b110), Subproblem(0b1000)),
            (Subproblem(0b110), Subproblem(0b11000)),
            (Subproblem(0b1110), Subproblem(0b10000)),
            (Subproblem(0b1), Subproblem(0b10)),
            (Subproblem(0b1), Subproblem(0b110)),
            (Subproblem(0b1), Subproblem(0b1110)),
            (Subproblem(0b1), Subproblem(0b11110)),
            (Subproblem(0b11), Subproblem(0b100)),
            (Subproblem(0b11), Subproblem(0b1100)),
            (Subproblem(0b11), Subproblem(0b11100)),
            (Subproblem(0b111), Subproblem(0b1000)),
            (Subproblem(0b111), Subproblem(0b11000)),
            (Subproblem(0b1111), Subproblem(0b10000)),
        ]

        generated_ccps = list(generate_ccp_pairs(chain_query))
        self.assertListEqual(generated_ccps, correct_ccps)


unittest.main(
    argv=["ignored", "-v"], defaultTest="EnumeratorTests", verbosity=2, exit=False
)